# 06_llm_prompt_design 

LLM prompt iteration on 20-post dev probe before full eval run. Keep cost under .

In [34]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np

root_dir = Path(os.getcwd()).parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
# Eigen scripts

from src.data import load_splits
from src.llm.prompts import build_prompt
from src.llm.openai_client import classify as classify_gpt
from src.llm.gemini_client import classify as classify_gemini

# Laad de API keys
load_dotenv()
print("API Keys geladen:", "GOOGLE_API_KEY" in os.environ)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
API Keys geladen: True


In [35]:
train_df, val_df, test_df = load_splits()

# Test op 20 posts
test_subset = train_df.sample(20, random_state=42)

# Haal de few-shot voorbeelden uit de rest van de training set
few_shot_examples = (
    train_df.drop(test_subset.index)
    .groupby("label_id")
    .apply(lambda g: g.sample(2, random_state=42))
    .reset_index(drop=True)[["text", "label"]]
    .to_dict("records")
)

print(f"Subset van {len(test_subset)} posts klaar voor test.")

Subset van 20 posts klaar voor test.


In [36]:
# Check de prompt voor de eerste post uit je subset
sample_post = test_subset.iloc[0]['text']
prompt_check = build_prompt(variant="few_shot", post=sample_post, few_shot_examples=few_shot_examples)

print("--- VOORBEELD PROMPT ---")
print(prompt_check)

--- VOORBEELD PROMPT ---
You are an assistant that classifies social media posts based on the severity of depression symptoms.
Classify the following post into one of four categories based on the described symptoms:
Choose one label that best fits the symptoms described in the post:
- minimum: no or minimal symptoms
- mild: mild symptoms, still functions mostly normally
- moderate: clear symptoms, noticeably reduced functioning
- severe: severe symptoms, significant suffering or dysfunction

Post: "I've been frustrated in the past by 2AM pulling it down and running too soon. But because I want to have my cake and eat it too, if on that 3rd down play he cut left, he's walking for a 1st down."
Label: "minimum"

Post: "But now, I can't. I literally have ONE therapist I can see. And ONE psychiatrist (who is actually a nurse practitioner). I have completely given up on getting the correct mental health treatment. I am doing the best I can."
Label: "minimum"

Post: "i rlly am tired of starti

CHATGPT TEST

In [23]:
prompt = build_prompt(variant="chain_of_thought", post=row["text"], few_shot_examples=few_shot_examples)
print("--- CHAIN OF THOUGHT PROMPT ---")
print(prompt)

--- CHAIN OF THOUGHT PROMPT ---
Je bent een assistent die sociale media posts classificeert op ernst van depressie.
Kies één label dat het beste past bij de symptomen beschreven in de post:
- minimum: geen of minimale symptomen
- mild: milde symptomen, functioneert nog grotendeels normaal
- moderate: duidelijke symptomen, merkbaar verminderd functioneren
- severe: ernstige symptomen, significant lijden of disfunctioneren

Post: "I've been frustrated in the past by 2AM pulling it down and running too soon. But because I want to have my cake and eat it too, if on that 3rd down play he cut left, he's walking for a 1st down."
Label: "minimum"

Post: "But now, I can't. I literally have ONE therapist I can see. And ONE psychiatrist (who is actually a nurse practitioner). I have completely given up on getting the correct mental health treatment. I am doing the best I can."
Label: "minimum"

Post: "i rlly am tired of starting over , i just wanna be happy"
Label: "mild"

Post: "Finished on my b

In [16]:
results = []

for i, (idx, row) in enumerate(test_subset.iterrows()):
    # We testen Chain of Thought omdat die het meest complex is
    prompt = build_prompt(variant="chain_of_thought", post=row["text"], few_shot_examples=few_shot_examples)
    
    # Roep de client aan
    res = classify_gpt(prompt, config={})
    
    # Sla resultaat op
    res['true_label'] = row['label']
    results.append(res)
    
    print(f"Post {i+1}/20: Model koos '{res['label']}' (Echt: {res['true_label']})")

test_results_df = pd.DataFrame(results)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

GEMINI TEST

In [ ]:
results = []

for i, (idx, row) in enumerate(test_subset.iterrows()):
    # We testen Chain of Thought omdat die het meest complex is
    prompt = build_prompt(variant="chain_of_thought", post=row["text"], few_shot_examples=few_shot_examples)
    
    # Roep de client aan
    res = classify_gemini(prompt, config={"model": "gemini-3.5-flash", "temperature": 0.0, "max_tokens":1600})
    
    # Sla resultaat op
    res['true_label'] = row['label']
    results.append(res)
    # print(res['label'])
    print(f"Post {i+1}/20: Model koos '{res['label']}' (Echt: {res['true_label']})")

test_results_df = pd.DataFrame(results)

Post 1/20: Model koos 'mild' (Echt: mild)
Post 2/20: Model koos 'minimum' (Echt: minimum)
Post 3/20: Model koos 'moderate' (Echt: severe)
Post 4/20: Model koos 'minimum' (Echt: minimum)
Post 5/20: Model koos 'minimum' (Echt: minimum)
Post 6/20: Model koos 'mild' (Echt: mild)
Post 7/20: Model koos 'minimum' (Echt: minimum)
Post 8/20: Model koos 'minimum' (Echt: minimum)
Post 9/20: Model koos 'minimum' (Echt: minimum)
Post 10/20: Model koos 'minimum' (Echt: minimum)
Post 11/20: Model koos 'mild' (Echt: minimum)
Post 12/20: Model koos 'minimum' (Echt: minimum)
Post 13/20: Model koos 'minimum' (Echt: minimum)
Post 14/20: Model koos 'moderate' (Echt: mild)
Post 15/20: Model koos 'minimum' (Echt: minimum)
Post 16/20: Model koos 'severe' (Echt: moderate)
Post 17/20: Model koos 'minimum' (Echt: minimum)
Post 18/20: Model koos 'mild' (Echt: mild)
Post 19/20: Model koos 'minimum' (Echt: mild)


In [18]:
avg_cost = test_results_df['cost_usd'].mean()
total_estimated = avg_cost * 1040 * 3 * 2 # 1040 posts, 3 varianten, 2 modellen

print(f"Gemiddelde kosten per call: ${avg_cost:.4f}")
print(f"Geschatte totale kosten: ${total_estimated:.2f}")

# Laat de eerste 5 resultaten zien inclusief de reasoning
test_results_df[['true_label', 'label', 'reasoning', 'cost_usd']].head()

Gemiddelde kosten per call: $0.0000
Geschatte totale kosten: $0.00


,true_label,label,reasoning,cost_usd
0,mild,unknown,None,0.0
1,minimum,unknown,None,0.0
2,severe,unknown,None,0.0
3,minimum,unknown,None,0.0
4,minimum,unknown,None,0.0
